In [ ]:
import heapq
import random
import math


SIZE = 70

START = (0, 0)
GOAL = (69, 69)


# ----------------------------------------------
# HEURISTIC
# ----------------------------------------------

def h_value(a, b):

    return abs(a[0] - b[0]) + abs(a[1] - b[1])


# ----------------------------------------------
# A* SEARCH
# ----------------------------------------------

def astar(grid, start, goal):

    directions = [
        (-1, 0),
        (1, 0),
        (0, -1),
        (0, 1)
    ]

    queue = []

    heapq.heappush(
        queue,
        (h_value(start, goal), 0, start)
    )

    distance = {
        start: 0
    }

    parent = {
        start: None
    }

    while queue:

        _, g, current = heapq.heappop(queue)

        if current == goal:

            result = []

            while current is not None:
                result.append(current)
                current = parent[current]

            return result[::-1]

        for dx, dy in directions:

            x = current[0] + dx
            y = current[1] + dy

            if not (0 <= x < SIZE and 0 <= y < SIZE):
                continue

            if grid[x][y] == 1:
                continue

            new_position = (x, y)

            new_distance = g + 1

            if (
                new_position not in distance
                or new_distance < distance[new_position]
            ):

                distance[new_position] = new_distance

                parent[new_position] = current

                f = (
                    new_distance +
                    h_value(new_position, goal)
                )

                heapq.heappush(
                    queue,
                    (f, new_distance, new_position)
                )

    return None


# ----------------------------------------------
# CREATE ENVIRONMENT
# ----------------------------------------------

def create_environment():

    grid = [
        [0 for _ in range(SIZE)]
        for _ in range(SIZE)
    ]

    for _ in range(750):

        cell = (
            random.randrange(SIZE),
            random.randrange(SIZE)
        )

        if cell != START and cell != GOAL:

            grid[cell[0]][cell[1]] = 1

    return grid


# ----------------------------------------------
# CREATE MOVING OBSTACLES
# ----------------------------------------------

def create_moving_obstacles(grid, count):

    obstacles = []

    while len(obstacles) < count:

        position = (
            random.randrange(SIZE),
            random.randrange(SIZE)
        )

        if (
            position != START
            and position != GOAL
            and grid[position[0]][position[1]] == 0
            and position not in obstacles
        ):

            obstacles.append(position)

    return obstacles


# ----------------------------------------------
# MOVE OBSTACLES
# ----------------------------------------------

def update_obstacles(grid, obstacles):

    directions = [
        (-1, 0),
        (1, 0),
        (0, -1),
        (0, 1)
    ]

    updated = []

    for obstacle in obstacles:

        x, y = obstacle

        random.shuffle(directions)

        moved = False

        for dx, dy in directions:

            nx = x + dx
            ny = y + dy

            if not (0 <= nx < SIZE and 0 <= ny < SIZE):
                continue

            if (nx, ny) == GOAL:
                continue

            if grid[nx][ny] == 0:

                updated.append((nx, ny))
                moved = True

                break

        if not moved:
            updated.append(obstacle)

    return updated


# ----------------------------------------------
# MAIN SIMULATION
# ----------------------------------------------

grid = create_environment()

moving_obstacles = create_moving_obstacles(
    grid,
    12
)

current = START

steps = 0
replans = 0
maximum_steps = 10000


while current != GOAL and steps < maximum_steps:

    # Mark moving obstacles as blocked
    for x, y in moving_obstacles:
        grid[x][y] = 1

    # Find safest current route
    route = astar(
        grid,
        current,
        GOAL
    )

    replans += 1

    if route is None:

        print("No safe path currently available.")
        break

    # Move only one step
    if len(route) > 1:

        next_cell = route[1]

    else:

        next_cell = GOAL

    # Clear previous obstacle locations
    for x, y in moving_obstacles:
        grid[x][y] = 0

    # Move obstacles
    moving_obstacles = update_obstacles(
        grid,
        moving_obstacles
    )

    # Check whether obstacle entered next cell
    if next_cell in moving_obstacles:

        print("Obstacle detected in planned path.")
        print("Re-planning...")

        continue

    # Move UGV
    current = next_cell
    steps += 1


# ----------------------------------------------
# RESULTS
# ----------------------------------------------

print("\n========== DYNAMIC UGV RESULTS ==========")

if current == GOAL:

    print("Goal reached successfully.")
    print("Total UGV movements:", steps)
    print("Total replanning operations:", replans)

else:

    print("Goal could not be reached.")
    print("UGV stopped at:", current)
    print("Total movements:", steps)
    print("Replanning operations:", replans)

Obstacle detected in planned path.
Re-planning...

========== DYNAMIC UGV RESULTS ==========
Goal reached successfully.
Total UGV movements: 146
Total replanning operations: 147
